# Document parsing

In [1]:
from docx import Document

doc = Document("Timetable-REV5 Hypotheses re press coverage of Hepatitis C in UK.docx")
doc

In [2]:
def extract_only_bullets(docx_path):
    doc = Document(docx_path)
    bullet_points = []
    
    for paragraph in doc.paragraphs:
        pPr = paragraph._element.pPr
        
        if pPr is not None:
            numPr = pPr.find('.//{http://schemas.openxmlformats.org/wordprocessingml/2006/main}numPr')
            
            if numPr is not None:
                # Get the numId
                numId = numPr.find('.//{http://schemas.openxmlformats.org/wordprocessingml/2006/main}numId')
                
                if numId is not None:
                    num_id = numId.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val')
                    
                    # Check if it's a bullet (not numbered)
                    if is_bullet_list(doc, num_id):
                        bullet_points.append(paragraph.text)
    
    return bullet_points

def is_bullet_list(doc, num_id):
    """Check if a numbering ID corresponds to a bullet list"""
    try:
        numbering_part = doc.part.numbering_part
        numbering_xml = numbering_part.element
        
        # Find the num element with this numId
        num = numbering_xml.find(f'.//{{{numbering_xml.nsmap["w"]}}}num[@{{{numbering_xml.nsmap["w"]}}}numId="{num_id}"]')
        
        if num is not None:
            # Get the abstractNumId
            abstractNumId = num.find(f'.//{{{numbering_xml.nsmap["w"]}}}abstractNumId')
            abstract_num_id = abstractNumId.get(f'{{{numbering_xml.nsmap["w"]}}}val')
            
            # Find the abstractNum definition
            abstractNum = numbering_xml.find(f'.//{{{numbering_xml.nsmap["w"]}}}abstractNum[@{{{numbering_xml.nsmap["w"]}}}abstractNumId="{abstract_num_id}"]')
            
            if abstractNum is not None:
                # Check the first level's numFmt
                lvl = abstractNum.find(f'.//{{{numbering_xml.nsmap["w"]}}}lvl[@{{{numbering_xml.nsmap["w"]}}}ilvl="0"]')
                numFmt = lvl.find(f'.//{{{numbering_xml.nsmap["w"]}}}numFmt')
                
                # Bullet lists use 'bullet' format, numbered use 'decimal', 'lowerLetter', etc.
                fmt_val = numFmt.get(f'{{{numbering_xml.nsmap["w"]}}}val')
                return fmt_val == 'bullet'
    except:
        pass
    
    return False

In [3]:
bullets = extract_only_bullets("Timetable-REV5 Hypotheses re press coverage of Hepatitis C in UK.docx")
bullets

['1950: The Haemophilia Society (UK) was formed.',
 'Late 1980s: Early Treatments: Interferon was the first antiviral treatment for hepatitis C.  significant side effects and lower cure rates.',
 'Late 1980s: Victims and their families first started calling for compensation, often on the grounds of medical negligence, as more evidence is gathered about link between blood products and HIV/hepatitis infections',
 '1987. The Birchgrove Group (often referred to as the BirchGrove Society in contexts of support), which supports people with haemophilia, HIV, and Hepatitis C, established in Wales by a small group of patients at the University Hospital of Wales (UHW) in Cardiff.  Early 1990s expanded to a national level. ',
 'November 1987: announcement of £10 million payment for haemophiliacs infected with HIV',
 '7 March 1988: British Liver Trust incorporated and established.  (Founded by hepatologist Professor Dame Sheila Sherlock, leading charity for adults, focusing on raising awareness, p

In [4]:
import re

date_pattern = r'\b(?:' \
    r'\d{1,2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+\d{4}|' \
    r'(?:Early|Mid|Late)\s+\d{4}s?|' \
    r'(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+\d{4}' \
    r')\b'

def extract_date(text):
    """Extract the date string from text"""
    match = re.search(date_pattern, text, re.IGNORECASE)
    if match:
        return match.group(0)
    return None

def remove_date(text, date_str):
    """Remove the date from the text and clean up extra whitespace"""
    cleaned = text.replace(date_str, '').strip()
    # Clean up any double spaces or extra punctuation left behind
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = re.sub(r'^\s*[-:]\s*', '', cleaned)  # Remove leading dash or colon
    return cleaned

def group_bullets_by_date_dict(bullet_list):
    """Create dictionary with dates as keys and associated text as values"""
    date_dict = {}
    current_date = None
    
    for bullet in bullet_list:
        date_str = extract_date(bullet)
        
        if date_str:
            # This bullet has a date - start a new entry
            current_date = date_str
            # Remove date from text and make it the first element
            text_without_date = remove_date(bullet, date_str)
            date_dict[current_date] = [text_without_date]
        else:
            # This bullet doesn't have a date - add to current date's list
            if current_date:
                date_dict[current_date].append(bullet)
    
    return date_dict


In [5]:
grouped = group_bullets_by_date_dict(bullets)
grouped, len(grouped)

({'Late 1980s': ['Victims and their families first started calling for compensation, often on the grounds of medical negligence, as more evidence is gathered about link between blood products and HIV/hepatitis infections'],
  'Early 1990s': ['1987. The Birchgrove Group (often referred to as the BirchGrove Society in contexts of support), which supports people with haemophilia, HIV, and Hepatitis C, established in Wales by a small group of patients at the University Hospital of Wales (UHW) in Cardiff. expanded to a national level.'],
  'November 1987': ['announcement of £10 million payment for haemophiliacs infected with HIV'],
  '7 March 1988': ['British Liver Trust incorporated and established. (Founded by hepatologist Professor Dame Sheila Sherlock, leading charity for adults, focusing on raising awareness, providing support, and campaigning for improved early detection and treatment of liver disease)'],
  '10 March 1988': ['Macfarlane Trust established',
   'April 21, 1989: ‘Discove

# Retrieval

In [6]:
import pandas as pd

df = pd.read_csv("./all_ratings731.csv")
def parse_date(val):
    for fmt in ('%Y-%m-%d', '%B %d, %Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError) as e:
            continue
    return pd.NaT  # fallback if nothing matches

df['Date'] = df['Date'].apply(lambda x: x.replace(" 00:00:00", "")).apply(parse_date)
df = df[["Title", "Date", "Author", "Newspaper", "Source", "newspaper_type", "chunks", "rating"]]

In [7]:
df

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks,rating
0,International Companies: Technology - Pounds 2...,1983-02-11,DAVID FISHLOCK,Financial Times,Factiva - HepC,broadsheet,BRITAIN'S biotechnologists are to have a new P...,0.2
1,Study confirms fears on spread of Aids,1984-08-31,ANDREW VEITCH,The Guardian,Lexis-Nexis - Times,broadsheet,Fears that the Aids virus is widespread among ...,0.2
2,Maternity unit where waste parades the corrido...,1984-10-20,No author in byline,The Guardian,Lexis-Nexis - Times,broadsheet,Dawn in the Rosie maternity unit at the 900-be...,0.2
3,Prisons inspector condemns sanitary conditions...,1984-10-24,AILEEN BALLANTYNE,The Guardian,Lexis-Nexis - Times,broadsheet,Gaol conditions fall short of the standard exp...,0.6
4,Financial Times Survey: China - Business - Gro...,1984-10-29,No author in byline,Financial Times,Factiva - HepC,broadsheet,CLIMATENorth China (including Beijing) - Winte...,0.0
...,...,...,...,...,...,...,...,...
8433,Wednesday \nEdition 1; \nNational Edition\n\n...,2016-08-10,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 10, 2016 Wednesda...",1.0
8434,Tuesday \nEdition 1; \nNational Edition\n\nDe...,2016-08-16,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 16, 2016 Tuesday ...",0.8
8435,Wednesday \nEdition 2; \nNational Edition\n\n...,2016-10-05,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 5, 2016 Wednesda...",1.0
8436,Friday \nEdition 1; \nIreland\n\nclassical Le...,2016-10-14,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 14, 2016 Friday ...",0.4


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("/Users/pnadel01/fp_models/google_embeddinggemma-300m", device="mps")

/Users/pnadel01/projects/time-bounded-search/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 7428.28it/s]


In [9]:
# test key: '29 August 2003'

sent_sim_template = "task: sentence similarity | query: {content}"

date_key_from_timetable = '29 August 2003'
associated_text = grouped[date_key_from_timetable]
query_embedding = model.encode(sent_sim_template.format(content=associated_text))
query_embedding.shape

(768,)

In [26]:
times = df[df.Newspaper == "The Times"]
times

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks,rating
7,Travel: How to stay healthy on your summer bre...,1985-07-06,ISABEL WALKER,The Times,Factiva - HepC,broadsheet,Holidays are one of the 30 most stressful even...,0.2
8,Family Money: Taking the sting from jabs / Ino...,1985-08-03,VIVIEN GOLDSMITH,The Times,Factiva - HepC,broadsheet,The series of jabs you will need if you are of...,0.2
9,A BREAKDOWN OF THE BREAKDOWNS (732),1985-08-14,JOHN GOODBODY,The Times,Factiva - HepC,broadsheet,Steve Cram1981Has a winter of injuries.August:...,0.0
10,Times Diary: Les-majeste (105),1985-08-30,PHS,The Times,Factiva - HepC,broadsheet,After years of pretending to find Barry Humphr...,0.4
13,Blood tests could fail to identify Aids virus ...,1985-09-30,No author in byline,The Times,Factiva - HepC,broadsheet,Aids-infected blood could fail to be detected ...,0.0
...,...,...,...,...,...,...,...,...
8433,Wednesday \nEdition 1; \nNational Edition\n\n...,2016-08-10,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 10, 2016 Wednesda...",1.0
8434,Tuesday \nEdition 1; \nNational Edition\n\nDe...,2016-08-16,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 16, 2016 Tuesday ...",0.8
8435,Wednesday \nEdition 2; \nNational Edition\n\n...,2016-10-05,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 5, 2016 Wednesda...",1.0
8436,Friday \nEdition 1; \nIreland\n\nclassical Le...,2016-10-14,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 14, 2016 Friday ...",0.4


In [27]:
key_dt = pd.to_datetime(date_key_from_timetable)
buffer = pd.DateOffset(years=1)

zone_of_interest = times[
    (times.Date >= key_dt - buffer) & # backward looking
    (times.Date <= key_dt + buffer)   # forward looking
]
zone_of_interest 

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks,rating
2141,Lawsuit reassurances fail to revive Weston - M...,2002-08-31,No author in byline,The Times,Factiva - HepC,broadsheet,REASSURING comments that Weston Medical's need...,0.2
2145,Body piercing craze 'is putting teenagers' hea...,2002-09-10,Sam Lister and Patrick Barkham.,The Times,Factiva - HepC,broadsheet,"Body piercings below the neck, once the preser...",0.6
2146,Gilt of the body politic - Cover story.,2002-09-11,Theodore Dalrymple.,The Times,Factiva - HepC,broadsheet,As tough new restrictions are proposed on body...,0.2
2147,The Times Vet - Pets.,2002-09-14,James Allcock.,The Times,Factiva - HepC,broadsheet,Q You answered a question about leg amputation...,0.0
2152,Liver failure death due to Ecstasy - In brief.,2002-09-17,No author in byline,The Times,Factiva - HepC,broadsheet,A 14-year-old girl suffered severe liver failu...,0.4
...,...,...,...,...,...,...,...,...
8315,", Saturday\n\n A new chapter",2004-05-15,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nMay 15, 2004, Saturday\n...",0.0
8316,", Tuesday\n\n NB\n\nSECTION: Features; Times2; 10",2004-05-25,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nMay 25, 2004, Tuesday\n\...",0.4
8317,", Saturday\n\n Shopping trips",2004-06-12,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 12, 2004, Saturday\...",0.0
8318,", Thursday\n\n So, anything to declare?",2004-07-29,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJuly 29, 2004, Thursday\...",0.2


In [21]:
doc_texts = [t.Title + "\n" + t.chunks for t in zone_of_interest.itertuples()]
doc_embeddings = model.encode([sent_sim_template.format(content=text) for text in doc_texts], show_progress_bar=True, device="mps")
doc_embeddings.shape

Batches: 100%|██████████| 5/5 [01:18<00:00, 15.67s/it]


(136, 768)

In [22]:
sim_scores = doc_embeddings @ query_embedding.T
rel_ids_and_scores =[(i, sim_scores[i]) for i in (sim_scores).argsort()[::-1]]
rel_ids_and_scores

[(np.int64(76), np.float32(0.7566831)),
 (np.int64(56), np.float32(0.7512807)),
 (np.int64(129), np.float32(0.7501491)),
 (np.int64(55), np.float32(0.7292455)),
 (np.int64(25), np.float32(0.7100658)),
 (np.int64(110), np.float32(0.6881811)),
 (np.int64(12), np.float32(0.67302406)),
 (np.int64(61), np.float32(0.67037684)),
 (np.int64(29), np.float32(0.66509414)),
 (np.int64(33), np.float32(0.65760314)),
 (np.int64(114), np.float32(0.64630866)),
 (np.int64(108), np.float32(0.6392649)),
 (np.int64(77), np.float32(0.6388498)),
 (np.int64(23), np.float32(0.63880193)),
 (np.int64(30), np.float32(0.63874966)),
 (np.int64(19), np.float32(0.6191523)),
 (np.int64(113), np.float32(0.61554486)),
 (np.int64(88), np.float32(0.6048112)),
 (np.int64(41), np.float32(0.6032578)),
 (np.int64(68), np.float32(0.5967107)),
 (np.int64(124), np.float32(0.5958388)),
 (np.int64(79), np.float32(0.5955116)),
 (np.int64(36), np.float32(0.5744972)),
 (np.int64(45), np.float32(0.5735587)),
 (np.int64(24), np.float32

In [10]:
import numpy as np

def scale_score(sim_scores, item_dates, target_date, scaler_func, date_weight):
    sim_scores = np.array(sim_scores)
    days_diffs = np.abs(np.array([(target_date - d).days for d in item_dates]))
    date_scores = np.array([scaler_func(d) for d in days_diffs])
    composite = (1 - date_weight) * sim_scores + date_weight * date_scores
    return composite, date_scores

def inverse_square(days_diff, max_days=365*2):
    normalized_diff = days_diff / max_days
    return 1 / (1 + normalized_diff ** 2)

In [25]:
composite, date_scores = scale_score(sim_scores, zone_of_interest.Date, key_dt, inverse_square, 0.5)
composite, date_scores 

(array([0.6331372 , 0.60922786, 0.55187218, 0.62564212, 0.61986936,
        0.64671886, 0.61361633, 0.60399447, 0.56903816, 0.60345979,
        0.69173124, 0.68657347, 0.76632209, 0.61485466, 0.6111096 ,
        0.64962247, 0.66642766, 0.68964796, 0.59193776, 0.75633707,
        0.72955951, 0.72195705, 0.62931905, 0.77566972, 0.74199701,
        0.8164813 , 0.65916149, 0.61189186, 0.66339318, 0.80328683,
        0.79562999, 0.66136626, 0.71856558, 0.80696522, 0.74178667,
        0.71299677, 0.77255995, 0.70150341, 0.71523413, 0.68159651,
        0.74144128, 0.78972552, 0.71902691, 0.69361529, 0.67146209,
        0.77896205, 0.67501672, 0.77019896, 0.70182797, 0.75365121,
        0.72516235, 0.70395476, 0.72241355, 0.72120268, 0.73512597,
        0.8646218 , 0.87563942, 0.77069203, 0.71821433, 0.75197838,
        0.71416697, 0.83422948, 0.75317481, 0.71042247, 0.74120478,
        0.69299982, 0.65620157, 0.70514932, 0.79465891, 0.70990356,
        0.673141  , 0.7313405 , 0.75861478, 0.75

In [28]:
from pprint import pprint

i = composite.argsort()[::-1][0]

print(i)
print(composite[i])
print(zone_of_interest.iloc[i].Title)
print(zone_of_interest.iloc[i].Date)
pprint(zone_of_interest.iloc[i].chunks)
print()
pprint(associated_text)

56
0.875639424240567
Cash hopes for hepatitis victims
2003-08-30 00:00:00
('THE campaign to win compensation payments for Scots infected with hepatitis '
 'C from infected NHS blood products took an important step forward yesterday '
 'as the Scottish Executive announced that the UK Government had agreed that '
 'such payments could be made by the devolved body.Malcolm Chisholm, '
 "Scotland's Health Minister, said that a similar scheme was being established "
 'in England by the Department of Health.A stumbling block to implementing any '
 'such scheme remains, however, because the Department of Work and Pensions in '
 'Whitehall has given no guarantee that it will not claw back social security '
 'payments from hepatitis victims given compensation.Under the proposed scheme '
 'in Scotland, ex gratia payments of Pounds 20,000 would be made to all those '
 'who have the hepatitis C virus as a result of infected NHS blood products. A '
 'further Pounds 25,000 would go to to those in an 

# Expand to dataset

In [ ]:
from pprint import pprint
def time_bounded_search(df, model, date_key_from_timetable, month_threshold=2, go_backwards=True, scale_fn=inverse_square, date_weight=0.5, sim_threshold=.75, newspaper=None):
    if newspaper:
        df = df[df.Newspaper == newspaper]
        print(f"Filtered dataframe to {len(df)} documents from newspaper: {newspaper}")
    associated_text = grouped[date_key_from_timetable]
    query_embedding = model.encode(sent_sim_template.format(content=associated_text))

    key_dt = pd.to_datetime(date_key_from_timetable)
    buffer = pd.DateOffset(months=month_threshold)

    if go_backwards:
        zone_of_interest = df[
            (df.Date >= key_dt - buffer) & # backward looking
            (df.Date <= key_dt + buffer)   # forward looking
        ]
    else:   
        zone_of_interest = df[
            (df.Date >= key_dt) &          # backward looking
            (df.Date <= key_dt + buffer)   # forward looking
        ]

    print(f"Processing {len(zone_of_interest)} documents in the zone of interest...")
    doc_texts = [t.Title + "\n" + t.chunks for t in zone_of_interest.itertuples()]
    doc_embeddings = model.encode([sent_sim_template.format(content=text) for text in doc_texts], show_progress_bar=True, device="mps")
    sim_scores = doc_embeddings @ query_embedding.T
    composite, _ = scale_score(sim_scores, zone_of_interest.Date, key_dt, scale_fn, date_weight)

    results = []
    for i in composite.argsort()[::-1]:
        score = composite[i]
        if score >= sim_threshold:
            results.append(zone_of_interest.iloc[i].to_dict())
    return results

def report_result(result, date_key_from_timetable):
    print("-"*40)
    print(f"Date from timetable: {date_key_from_timetable}")
    print(f"Text from timetable:")
    pprint(grouped[date_key_from_timetable])
    print("-"*40)
    print(f"Title: {result['Title']}")
    print(f"Date: {result['Date']}")
    print(f"Newspaper: {result['Newspaper']}")
    pprint(f"{result['chunks']}")
    print(f"Rating: {result['rating']*5}")
    print("-"*40)

In [46]:
grouped.keys()

dict_keys(['Late 1980s', 'Early 1990s', 'November 1987', '7 March 1988', '10 March 1988', 'July 1988', 'June 1989', '29 January 1990', 'April 1991', '3 May 1991', 'Sept 1991', '17 February 1992', '27 April 1992', '29 March 1993', 'May 1993', '11 July 1995', '26 March 2001', 'August 2001', 'June 2002', '20 January 2003', '29 August 2003', 'January 2004', '25 March 2004', 'January 2006', 'July 2006', '19 February 2007', 'March 2007', '23 April 2008', 'February 2009', 'March 2009', '20 May 2009', 'April 2010', '14 October 2010', 'January 2011', '8 March 2011', '22 June 2011', 'May 2011', 'October 2011', 'November 2013', 'December 2013', 'April 2014', 'August 2014', 'October 2014', 'March 2015', '25 March 2015', 'October 2015', 'November 2015', 'June 2016', 'January 2017', '11 July 2017', 'August 2017', 'May 2018', '2 July 2018', '5 October 2020', '10 December 2020', '20 May 2024', '31 March 2025', '9 July 2025', '21 July 2025', '30 October 2025', 'November 2025', '31 December 2025', '22 J

In [114]:
date_key = '19 February 2007'

results = time_bounded_search(
    df,
    model,
    date_key_from_timetable=date_key,
    go_backwards=False,
    month_threshold=12,
    newspaper="The Times"
)

Filtered dataframe to 1298 documents from newspaper: The Times
Processing 69 documents in the zone of interest...


Batches: 100%|██████████| 3/3 [03:31<00:00, 70.60s/it] 


In [115]:
len(results)

17

In [116]:
report_result(results[0], date_key)

----------------------------------------
Date from timetable: 19 February 2007
Text from timetable:
['Lord Archer (independent) Inquiry announced. On infection of haemophiliacs '
 'and others with HIV and Hepatitis C via contaminated blood']
----------------------------------------
Title: Contaminated blood inquiry
Date: 2007-02-19 00:00:00
Newspaper: The Times
('An independent public inquiry is to be held into the supply to haemophiliacs '
 'of contaminated NHS blood (a correspondent writes).The Labour peer Lord '
 'Archer of Sandwell, a former Solicitor-General, is to conduct the inquiry '
 'after a campaign by Lord Morris of Manchester, president of the Haemophilia '
 'Society and a former Minister for the Disabled, who said that 1,757 '
 'haemophilia patients who were exposed to HIV and/or hepatitis C-contaminated '
 'NHS blood and blood products had died since being infected. "Many more are '
 'now terminally ill," the peer claimed.Lord Morris said that, of 4,670 such '
 'patients